# Baseline — Mini Enefit: Solar Prosumer Energy Forecasting

**Competition:** predict hourly electricity **consumption and production** for Estonian
prosumer groups (a mini version of the Kaggle Enefit competition).
Each row is one hour for a segment defined by county, business/customer status,
product type, and whether the row is consumption or production.

- **Task:** regression — predict `target` for every row of `test.csv`
- **Metric:** `score = 1 / (1 + MAE)` (higher is better, perfect = 1.0)
- **Kaggle link:** _TODO: add link_

This baseline trains a gradient-boosted-trees model on the provided features with a
**time-based validation split** (always validate on the *latest* period in time-series
problems — a random split would leak the future into training).

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

DATA_DIR = "."   # folder containing train.csv / test.csv

train = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=["datetime"])
test  = pd.read_csv(f"{DATA_DIR}/test.csv",  parse_dates=["datetime"])
print(train.shape, test.shape)

(285744, 42) (21984, 41)


In [2]:
# Drop rows with missing target, pick feature columns (everything numeric except ids/target)
train = train.dropna(subset=["target"]).reset_index(drop=True)

drop_cols = {"id", "target", "datetime", "date", "data_block_id"}
features = [c for c in train.columns if c not in drop_cols and train[c].dtype != object]
print(len(features), "features")

37 features


In [3]:
# Time-based validation: last 14 days = validation
cutoff = train["datetime"].max() - pd.Timedelta(days=14)
tr, va = train[train["datetime"] <= cutoff], train[train["datetime"] > cutoff]
print("train:", len(tr), "valid:", len(va))

model = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.08, num_leaves=63, random_state=0, verbose=-1)
model.fit(tr[features], tr["target"])

pred_va = np.clip(model.predict(va[features]), 0, None)   # target is never negative
mae = mean_absolute_error(va["target"], pred_va)
print(f"Validation MAE   : {mae:.3f}")
print(f"Validation score : {1/(1+mae):.5f}   (competition metric)")

train: 240348 valid: 45264


Validation MAE   : 67.090
Validation score : 0.01469   (competition metric)


In [4]:
# Retrain on ALL data and predict the test set
model.fit(train[features], train["target"])
test_pred = np.clip(model.predict(test[features]), 0, None)

sub = pd.DataFrame({"id": test["id"], "target": test_pred})
sub.to_csv("submission.csv", index=False)
sub.head()

,id,target
0,test_000000,3.344808
1,test_000001,137.186672
2,test_000002,1.411001
3,test_000003,95.533409
4,test_000004,11.364455


## Ideas to improve

- Use **LightGBM / XGBoost / CatBoost** with tuned parameters and early stopping.
- Train **two separate models**: one for consumption rows, one for production rows —
  they behave very differently (production follows solar radiation, consumption
  follows hour/weekday patterns).
- Add more lag features and rolling means per `prediction_unit_id`.
- Normalize the target by `installed_capacity` for production rows.
- Ensemble several models and seeds.